# Mage-Flow trainer: data, latents and the remote GPU for the laptop GUI

Run **Settings** and **Install** first every session, then only the sections the machine needs:

| Section | Where | When |
| --- | --- | --- |
| 1. Prepare the dataset from the library | the Studio server | after each training-layout export |
| 2. Models | every machine that caches or trains | once per machine (re-running only checks) |
| 3. Download the latent cache | the training machine | each session on a fresh machine |
| 4. Write the configs | every machine | after 1 or 3 |
| 5. Cache latents and captions | the Studio server (its A4000s) | once per dataset |
| 6. Upload the latent cache | the Studio server | after 5 |
| 7. Train | the training machine (or the server) | |
| 8. Publish the dataset (optional) | the Studio server | once per release |

The plan this follows: the images never leave the Studio server. It caches latents where
GPU time is cheap, uploads only the latents and captions, and the expensive
training machine downloads just that: no images and no VAE pass (it encodes the captions itself at startup, which
is quick there; their cache is far too big to move). Section 8 publishes the images
separately, as streamable WebDataset shards, if you open-source the dataset.
Tokens are never written into this notebook: log in with the cell in Install, or set `HF_TOKEN`.
See `docs/remote-training.md`.

In [ ]:
# ---- Settings: edit, then run this cell first every session ----
import os
WORK = "/workspace"                         # trainer, models, training output (Studio JupyterHub: "/home/jovyan")
DATA_ROOT = "/root"                         # the BIG disk: dataset, latents, caption cache (on pods often not /workspace;
                                            # Studio JupyterHub: "/home/jovyan")
TRAINER_DIR = f"{WORK}/mage-flow-trainer"
MODELS_DIR = f"{WORK}/models"
DATA_DIR = f"{DATA_ROOT}/data/magetrail-v04-pilot"   # dataset root: folders.csv + <group>/<artist>/ folders
OUTPUT_DIR = f"{WORK}/output"

CACHE_REPO = "RicemanT/magetrail-v04-pilot-latents"     # private HF dataset: latents + captions + layout, no images
DATASET_REPO = "RicemanT/magetrail-v04-pilot"           # section 8 only: the published images as WebDataset shards

BASE_MODEL_REPO = "mage-flow-community/Mage-Flow"      # the community copy (Microsoft took the original down)
TRANSFORMER_REPO = "RicemanT/MageTrail"
TRANSFORMER_FILE = "V0.3/MageTrail-V0.3.safetensors"    # the checkpoint the finetune starts from (architecture embedded)
FLUX2_VAE_REPO = "black-forest-labs/FLUX.2-dev"         # the FLUX.2 VAE the latents are made with (gated: accept once)
FLUX2_VAE_FILES = "vae/*"                               # a diffusers vae/ folder, or one diffusers-layout .safetensors
VAE_PATH = f"{MODELS_DIR}/flux2-vae/vae"                # where FLUX2_VAE_FILES lands: that folder, or the file's path

GPUS = "0,1,2,3"                            # GPUs for caching here
os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")   # faster up/downloads (huggingface_hub uses Xet)
MODEL_PATH = f"{MODELS_DIR}/mage-flow"
TRANSFORMER_PATH = f"{MODELS_DIR}/{TRANSFORMER_FILE}"

import shutil   # which disk is which: the data needs the big one
for label, path in (("WORK", WORK), ("DATA_ROOT", DATA_ROOT)):
    try:
        usage = shutil.disk_usage(path)
        print(f"{label:9} {path:20} {usage.free / 1e9:8.0f} GB free of {usage.total / 1e9:.0f} GB")
    except OSError:
        print(f"{label:9} {path:20} (not found)")

In [ ]:
# ---- Install: the trainer (once per machine), Hugging Face tools, login ----
import os
os.makedirs(WORK, exist_ok=True)
%cd {WORK}
!git clone https://github.com/RicemanT/mage-flow-trainer || (cd mage-flow-trainer && git pull)
%cd {TRAINER_DIR}
!bash install.sh   # picks a Python that can make a venv; INSTALL_PYTHON=/path/to/python to choose
!venv/bin/python -m pip install -q -r requirements-tracking.txt   # only if you use wandb / TensorBoard / Trackio
%pip install -q "huggingface_hub>=1.0"

from huggingface_hub import login, whoami
try:
    print("Hugging Face:", whoami()["name"])
except Exception:
    login()   # a token with write access; huggingface_hub stores it, this notebook never does

## 1. Prepare the dataset from the library (on the Studio server)

In the Studio: **Planner → 5. Download → Export run → Export training layout**. This cell builds `DATA_DIR` from that
export: its files (`folders.csv`, configs, ...) at the root, and every `<group>/<artist>` folder it lists with its
images, `.txt` tags and `_nl.txt` captions **hard-linked** from the library: no copy, no extra disk space, and the
latents written next to them stay out of the library. Run it again after more curation or caption edits: it
refreshes changed files and removes images (and their latents) that left the library.

In [ ]:
LIBRARY_IMAGES = "/home/jovyan/Artist-Collection-Dataset/images"
TRAINING_EXPORT = "/home/jovyan/Artist-Collection-Dataset/planner/exports/run-3/training"   # the export's folder

import csv, glob, os, shutil
KEEP_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif", ".txt"}   # images, tags, _nl captions

def place(source, target):
    """Hard-link source to target (a copy if linking is impossible); keep an identical link as it is."""
    if os.path.exists(target):
        if os.path.samefile(source, target):
            return False
        os.remove(target)
    try:
        os.link(source, target)
    except OSError:
        shutil.copy2(source, target)
    return True

os.makedirs(DATA_DIR, exist_ok=True)
for entry in os.scandir(TRAINING_EXPORT):   # the export's files only (not .ipynb_checkpoints and the like)
    if entry.is_file() and not entry.name.startswith("."):
        shutil.copy2(entry.path, f"{DATA_DIR}/{entry.name}")
with open(f"{DATA_DIR}/folders.csv", encoding="utf-8") as fh:
    folders = [row["path"] for row in csv.DictReader(fh) if int(float(row.get("images") or 0))]
placed = removed = 0
for folder in folders:
    source_dir, target_dir = f"{LIBRARY_IMAGES}/{folder}", f"{DATA_DIR}/{folder}"
    os.makedirs(target_dir, exist_ok=True)
    wanted = {entry.name for entry in os.scandir(source_dir)
              if entry.is_file() and os.path.splitext(entry.name)[1].lower() in KEEP_EXT}
    for name in wanted:
        placed += place(f"{source_dir}/{name}", f"{target_dir}/{name}")
    for entry in os.scandir(target_dir):   # gone from the library: drop it, and its latents
        stem, ext = os.path.splitext(entry.name)
        if ext.lower() in KEEP_EXT and entry.name not in wanted:
            os.remove(entry.path)
            removed += 1
            for latent in glob.glob(f"{target_dir}/{glob.escape(stem)}_*_trainer.safetensors"):
                os.remove(latent)
print(f"{len(folders)} artist folders in {DATA_DIR} · {placed} files linked or updated · {removed} removed")

## 2. Models

The text encoder (Qwen3-VL-4B, with its tokenizer files) comes from the Mage-Flow repository (its base transformer
and Mage-VAE are skipped). The latents use the **FLUX.2 VAE** (`flux2_vae = true` in the configs), in diffusers format:
a `vae/` folder with `config.json`, or a `.safetensors` with diffusers key names. ComfyUI's single-file
`flux2-vae.safetensors` does not load here, and the cell stops if it gets one. The transformer is your checkpoint (its architecture is embedded in the file, so no config is needed). Already-downloaded files are
only checked, so re-running is quick.

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download
snapshot_download(BASE_MODEL_REPO, local_dir=MODEL_PATH,
                  ignore_patterns=["transformer/*.safetensors", "vae/*", "assets/*"])   # ~9 GB: the text encoder
snapshot_download(FLUX2_VAE_REPO, local_dir=f"{MODELS_DIR}/flux2-vae", allow_patterns=[FLUX2_VAE_FILES])
hf_hub_download(TRANSFORMER_REPO, TRANSFORMER_FILE, local_dir=MODELS_DIR)

# The FLUX.2 VAE must be diffusers format; ComfyUI's legacy single file needs ComfyUI itself to load.
if os.path.isfile(VAE_PATH):
    from safetensors import safe_open
    with safe_open(VAE_PATH, framework="pt") as fh:
        keys = set(fh.keys())
    if "bn.running_mean" in keys and "encoder.quant_conv.weight" in keys:
        raise ValueError(f"{VAE_PATH} is ComfyUI's FLUX.2 VAE layout: use a diffusers-format FLUX.2 VAE (a vae/ folder)")
elif not os.path.isfile(f"{VAE_PATH}/config.json"):
    raise FileNotFoundError(f"No FLUX.2 VAE at {VAE_PATH}: check FLUX2_VAE_REPO / FLUX2_VAE_FILES / VAE_PATH")
print("model_path       =", MODEL_PATH)
print("transformer_path =", TRANSFORMER_PATH)
print("vae_path         =", VAE_PATH, "(FLUX.2)")

## 3. Download the latent cache (on the training machine)

Latents, captions and the layout files from `CACHE_REPO`; no images. The caption-embedding cache is built here at the
first training start (a few minutes on big GPUs). Training reads the
latents (`dataset.source = "auto"` uses them when a folder has no images) and the captions beside them. Skip this
section on the Studio server, which already has everything from sections 1 and 5.

In [ ]:
snapshot_download(CACHE_REPO, repo_type="dataset", local_dir=DATA_DIR)
print("latent cache in", DATA_DIR)

## 4. Write the configs for this machine

Fills the `@...@` placeholders of the exported `mageflow-*.toml` with this machine's paths and saves them as
`configs/magetrail/*.toml` in the trainer. For the 1024 stage, set `STAGE_512_CHECKPOINT` to the 512 stage's final
checkpoint first (a `.safetensors` file under `OUTPUT_DIR/<run_name>/`). Open the files in the GUI to adjust
epochs, learning rates or prompts; with jobs mode the GUI edits the config it sends.

In [ ]:
import glob, re
STAGE_512_CHECKPOINT = ""        # e.g. f"{OUTPUT_DIR}/magetrail-run1-512/<final checkpoint>.safetensors"

values = {"@MODEL_PATH@": MODEL_PATH, "@TRANSFORMER_PATH@": TRANSFORMER_PATH, "@VAE_PATH@": VAE_PATH,
          "@DATA_DIR@": DATA_DIR, "@OUTPUT_DIR@": OUTPUT_DIR}
if STAGE_512_CHECKPOINT:
    values["@STAGE_512_CHECKPOINT@"] = STAGE_512_CHECKPOINT
target = f"{TRAINER_DIR}/configs/magetrail"
os.makedirs(target, exist_ok=True)
CONFIGS = {}
for source in sorted(glob.glob(f"{DATA_DIR}/mageflow-*.toml")):
    text = open(source, encoding="utf-8").read()
    for key, value in values.items():
        text = text.replace(key, value)
    name = os.path.basename(source)
    CONFIGS[name.removeprefix("mageflow-").removesuffix(".toml")] = f"{target}/{name}"
    with open(f"{target}/{name}", "w", encoding="utf-8") as fh:
        fh.write(text)
    left = sorted(set(re.findall(r"@[A-Z0-9_]+@", text)))
    print(f"{target}/{name}" + (f"   still to fill: {', '.join(left)}" if left else ""))

## 5. Cache latents and captions (on the Studio server, after 1, 2 and 4)

Encodes every listed image with the FLUX.2 VAE at each stage's resolution (`*_WWWWxHHHH_trainer.safetensors` beside the
image); already-cached files are skipped, so an interrupted run resumes. The second cell (caption-embedding cache) is
only for training on this same machine: that cache is far too big to upload (about 80 GB for the pilot).
Leave out GPUs that other work (the Studio's image analysis) is using.

In [ ]:
for stage, config in CONFIGS.items():
    print(f"=== latents for {stage}")
    !cd {TRAINER_DIR} && venv/bin/python -m trainer.tools.cache_latents cache-config {config} --devices {GPUS}

In [ ]:
# OPTIONAL, only to train on THIS machine: the caption-embedding cache is huge (whole Qwen3-VL hidden states:
# ~80 GB for the 6.5k-image pilot at 8 variations, terabytes for the full dataset), so it is not uploaded.
# The training machine builds its own at startup, which takes minutes on big GPUs.
first = CONFIGS.get("512") or next(iter(CONFIGS.values()))   # any stage works; 512 has no checkpoint left to fill
n = len([g for g in GPUS.split(",") if g.strip()])
multi = "--multi_gpu" if n > 1 else ""   # a saved single-process Accelerate config would otherwise win
!cd {TRAINER_DIR} && CUDA_VISIBLE_DEVICES={GPUS} venv/bin/accelerate launch {multi} --num_processes {n} -m trainer.training.train {first} --text-cache-only

## 6. Upload the latent cache

Uploads the latents, the `.txt`/`_nl.txt` captions and the layout files from `DATA_DIR` (never the images, and not the
caption-embedding cache, which the training machine builds itself) to `CACHE_REPO`. The first cell checks that every image has its latents for both stages and its
captions, so the training machine misses nothing.

In [ ]:
# Every listed image needs one latent per stage and its captions before the cache goes up.
import csv, glob, os
IMAGE_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif"}
stages = len(CONFIGS)
missing_latents, missing_captions, images = [], [], 0
with open(f"{DATA_DIR}/folders.csv", encoding="utf-8") as fh:
    folders = [row["path"] for row in csv.DictReader(fh) if int(float(row.get("images") or 0))]
for folder in folders:
    for entry in os.scandir(f"{DATA_DIR}/{folder}"):
        stem, ext = os.path.splitext(entry.name)
        if ext.lower() not in IMAGE_EXT:
            continue
        images += 1
        # one latent per stage, but a small image (no upscaling) gets the same bucket, so the same file, in every stage
        if not glob.glob(f"{DATA_DIR}/{glob.escape(folder)}/{glob.escape(stem)}_*_trainer.safetensors"):
            missing_latents.append(f"{folder}/{entry.name}")
        if not os.path.exists(f"{DATA_DIR}/{folder}/{stem}.txt"):
            missing_captions.append(f"{folder}/{entry.name}")
print(f"{images} images in {len(folders)} folders · {len(missing_latents)} without a latent"
      f" · {len(missing_captions)} without a .txt")
for label, items in (("no latents", missing_latents), ("no captions", missing_captions)):
    if items:
        print(f"  {label}: {items[:5]}{' ...' if len(items) > 5 else ''}")

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(CACHE_REPO, repo_type="dataset", private=True, exist_ok=True)
# huggingface_hub 2.x: upload_folder commits a large folder in batches and resumes when re-run (with hf_xet)
api.upload_folder(repo_id=CACHE_REPO, repo_type="dataset", folder_path=DATA_DIR, commit_message="Latent cache",
                        allow_patterns=["*_trainer.safetensors", "*.txt", "*.csv", "*.toml", "*.json"],
                        ignore_patterns=[".*", "*/.*", "*.lock"])
print("Uploaded to https://huggingface.co/datasets/" + CACHE_REPO)

## 7. Train

**A. Jobs mode** (the GUI's Start Training runs here): leave the cell running and paste the printed link into the
GUI's Remote bar, then Connect. Load `configs/magetrail/mageflow-512.toml` in the GUI (or set the same paths:
`model_path`, `transformer_path`, `subsets_file`, `caption_cache_path`). Logs and graphs appear in the GUI; Stop /
Save now / Save & stop work from there.

In [ ]:
!cd {TRAINER_DIR} && venv/bin/python -m trainer.remote serve
# Own address instead of a Cloudflare tunnel, e.g. a RunPod proxy port:
# !cd {TRAINER_DIR} && venv/bin/python -m trainer.remote serve --tunnel none --url https://<pod-id>-8765.proxy.runpod.net

**B. Notebook mode:** the GUI only sends the config (Start Training), this cell receives it, the next one trains it.

In [ ]:
!cd {TRAINER_DIR} && venv/bin/python -m trainer.remote receive

In [ ]:
# The config the previous cell printed, or one of CONFIGS from section 4. Caches what is missing, then trains.
CONFIG = CONFIGS.get("512", "configs/remote/mageflow.toml")
!cd {TRAINER_DIR} && venv/bin/python -m trainer.remote run {CONFIG}
# Checkpoint on demand from another cell:  !touch {OUTPUT_DIR}/<run_name>/save   (or save_quit to stop)

## 8. Publish the dataset (optional, on the Studio server)

For open-sourcing the images: **WebDataset** tar shards of about 1 GB, which Hugging Face streams natively
(`load_dataset(repo, streaming=True)` or the `webdataset` library), instead of hundreds of thousands of loose files.
Each sample is one image with `tags.txt`, `caption.txt` and a `json` of its artist, trigger, group and folder. A
dataset card and the layout files go with them. Shards are written to `PUBLISH_DIR` first (about the dataset's size),
then uploaded; re-running the cell resumes an interrupted upload. Set `PUBLIC = True` to make the repo public.

In [ ]:
import csv, io, json, os, tarfile
from huggingface_hub import HfApi
PUBLISH_DIR = f"{WORK}/publish/{DATASET_REPO.split('/')[-1]}"
SHARD_BYTES = 1_000_000_000
PUBLIC = False
IMAGE_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif"}

with open(f"{DATA_DIR}/folders.csv", encoding="utf-8") as fh:
    rows = [row for row in csv.DictReader(fh) if int(float(row.get("images") or 0))]
os.makedirs(f"{PUBLISH_DIR}/data", exist_ok=True)
shard, size, samples, shards = None, 0, 0, 0

def add(tar, name, data):
    info = tarfile.TarInfo(name)
    info.size = len(data)
    tar.addfile(info, io.BytesIO(data))

for row in rows:
    folder = f"{DATA_DIR}/{row['path']}"
    for entry in sorted(os.scandir(folder), key=lambda e: e.name):
        stem, ext = os.path.splitext(entry.name)
        if ext.lower() not in IMAGE_EXT:
            continue
        if shard is None or size >= SHARD_BYTES:
            if shard:
                shard.close()
            shard = tarfile.open(f"{PUBLISH_DIR}/data/shard-{shards:05d}.tar", "w")
            shards, size = shards + 1, 0
        key = f"{row['path'].replace('/', '__')}__{stem}"   # WebDataset keys: one per sample, no dots
        meta = {"artist": row.get("artist"), "trigger": row.get("trigger"), "site": row.get("site"),
                "group": row["path"].split("/")[0], "folder": row["path"], "file": entry.name}
        image = open(entry.path, "rb").read()
        add(shard, f"{key}{ext.lower()}", image)
        for suffix, field in ((".txt", "tags.txt"), ("_nl.txt", "caption.txt")):
            if os.path.exists(f"{folder}/{stem}{suffix}"):
                add(shard, f"{key}.{field}", open(f"{folder}/{stem}{suffix}", "rb").read())
        add(shard, f"{key}.json", json.dumps(meta, ensure_ascii=False).encode())
        size += len(image)
        samples += 1
if shard:
    shard.close()
for name in ("folders.csv", "folders.json", "subsets.toml"):
    if os.path.exists(f"{DATA_DIR}/{name}"):
        open(f"{PUBLISH_DIR}/{name}", "wb").write(open(f"{DATA_DIR}/{name}", "rb").read())
with open(f"{PUBLISH_DIR}/README.md", "w", encoding="utf-8") as fh:
    fh.write(f"""---
configs:
- config_name: default
  data_files: "data/*.tar"
---
# {DATASET_REPO.split('/')[-1]}

{samples} images from {len(rows)} artists in {shards} WebDataset shards. Each sample has the image, `tags.txt`
(booru tags, artist trigger first), `caption.txt` (natural-language caption) and a `json` with its artist, trigger,
group and folder. `folders.csv` lists the artist folders with their training repeats.

```python
from datasets import load_dataset
ds = load_dataset("{DATASET_REPO}", streaming=True, split="train")
```
""")
print(f"{samples} samples in {shards} shards under {PUBLISH_DIR}")
api = HfApi()
api.create_repo(DATASET_REPO, repo_type="dataset", private=not PUBLIC, exist_ok=True)
api.upload_folder(repo_id=DATASET_REPO, repo_type="dataset", folder_path=PUBLISH_DIR, commit_message="Dataset shards",
                  ignore_patterns=[".*", "*/.*"])
print("Published to https://huggingface.co/datasets/" + DATASET_REPO)